# 📅 Day 14 — Mock Interview Practice

**Week 2 · Day 7 (final day of Week 2) · Level: INTERMEDIATE → ADVANCED**

Today wraps up Week 2: multi-step problems, real-world scenarios, and optimization. 18 questions across 8 categories — 9 [INTERMEDIATE] + 9 [ADVANCED].

| # | Category | Questions | Level |
| --- | --- | --- | --- |
| 1 | SQL Advanced | Q1-Q2 | INTERMEDIATE → ADVANCED |
| 2 | Python — Loop/Function/List/Dict | Q3-Q4 | INTERMEDIATE → ADVANCED |
| 3 | PySpark Coding | Q5-Q6 | INTERMEDIATE → ADVANCED |
| 4 | Spark Optimization & Deep Spark | Q7-Q8 | INTERMEDIATE → ADVANCED |
| 5 | ETL — Failures, Late Data, Schema, Fault Tolerance | Q9-Q11 | INTERMEDIATE → ADVANCED |
| 6 | Data Modeling | Q12-Q13 | INTERMEDIATE → ADVANCED |
| 7 | Snowflake | Q14-Q15 | INTERMEDIATE → ADVANCED |
| 8 | Monitoring, Logging & Orchestration | Q16-Q18 | INTERMEDIATE → ADVANCED |

> Week 3 starts tomorrow: ADVANCED / [HARD] product-company level.

---

## Q1 [INTERMEDIATE] SQL Advanced — Funnel Drop-off Analysis with Conditional Aggregation

**Scenario**: You are given an e-commerce `events` table: `user_id`, `event_name` (`view`, `cart`, `purchase`), `event_ts` (epoch). The product team wants a strict 3-step purchase funnel.

**Requirements**:
1. A user progresses to a step only if the step's event happened **after** the previous step's event (strict sequential ordering — a `cart` recorded *before* the first `view` does not count).
2. Report users reaching each step and the step-to-step conversion percentages.
3. Watch out for users who **skip steps** (view → purchase) — they count for step 1 only in a strict funnel.

**Sample data design** (verify your query against these expectations):

| User | Events | Expected strict funnel |
| --- | --- | --- |
| u1 | view(1) → cart(2) → purchase(3) | all 3 steps |
| u2 | view(1) → cart(2) | 2 steps |
| u3 | view(1) | 1 step |
| u4 | cart(1) → view(2) → purchase(3) | view only (cart before view) |
| u5 | view(1) → purchase(2) | view only (skips cart) |
| u6 | view(1) → cart(5) → purchase(3) | view + cart (purchase before cart) |

Expected output: `step1_view=6, step2_cart=3, step3_purchase=1, view_to_cart=50.0%, cart_to_purchase=33.3%`.

In [0]:
# Solution Q1 — strict sequential funnel via conditional aggregation
import pyspark.sql.functions as F

spark.sql("DROP TABLE IF EXISTS workspace.default.d14_events")
events_data = [
    ("u1", "view", 1), ("u1", "cart", 2), ("u1", "purchase", 3),
    ("u2", "view", 1), ("u2", "cart", 2),
    ("u3", "view", 1),
    ("u4", "cart", 1), ("u4", "view", 2), ("u4", "purchase", 3),   # out of order
    ("u5", "view", 1), ("u5", "purchase", 2),                      # skips cart
    ("u6", "view", 1), ("u6", "cart", 5), ("u6", "purchase", 3),   # purchase before cart
]
events = spark.createDataFrame(events_data, ["user_id", "event_name", "event_ts"])
events.write.mode("overwrite").saveAsTable("workspace.default.d14_events")

q1 = spark.sql("""
WITH steps AS (
  -- first occurrence of each step per user
  SELECT user_id,
         MIN(CASE WHEN event_name = 'view'     THEN event_ts END) AS t_view,
         MIN(CASE WHEN event_name = 'cart'     THEN event_ts END) AS t_cart,
         MIN(CASE WHEN event_name = 'purchase' THEN event_ts END) AS t_purchase
  FROM workspace.default.d14_events
  GROUP BY user_id
),
strict AS (
  -- keep a step only if it happens AFTER the previous step
  SELECT user_id,
         t_view,
         CASE WHEN t_cart > t_view THEN t_cart END AS t_cart_ok,
         CASE WHEN t_cart > t_view AND t_purchase > t_cart THEN t_purchase END AS t_purchase_ok
  FROM steps
)
SELECT
  COUNT(*)               AS step1_view,
  COUNT(t_cart_ok)       AS step2_cart,        -- COUNT(col) ignores NULLs
  COUNT(t_purchase_ok)   AS step3_purchase,
  ROUND(100 * COUNT(t_cart_ok) / COUNT(*), 1)               AS view_to_cart_pct,
  ROUND(100 * COUNT(t_purchase_ok) / COUNT(t_cart_ok), 1)  AS cart_to_purchase_pct
FROM strict
""")
q1.show()

r = q1.first()
assert (r.step1_view, r.step2_cart, r.step3_purchase) == (6, 3, 1), r
print("Q1 OK — funnel: 6 view → 3 cart → 1 purchase")

# Interview notes:
# - COUNT(col) counts non-nulls — the CASE-based nulling trick is the classic funnel pattern.
# - MIN(ts) per step + ordering comparisons enforce STRICT sequencing; drop the
#   comparisons for a loose funnel ("ever did each step") and the numbers will grow.
# - For longer funnels, generalize with a window LEAD over a per-user event sequence, or
#   sequence matching (MATCH_RECOGNIZE) in Databricks SQL.

## Q2 [ADVANCED] SQL Advanced — Approximate Distinct Count at Scale

**Scenario**: A dashboard counts daily active users over a 10-billion-row events table. `COUNT(DISTINCT user_id)` takes minutes and shuffles terabytes; the business is fine with ±2% error.

**Requirements**:
1. Explain why exact `COUNT(DISTINCT)` is expensive in a distributed engine (shuffle of all distinct keys, memory pressure) and how **HyperLogLog (HLL)** sketches solve it.
2. Demonstrate `APPROX_COUNT_DISTINCT(col)` (default relative standard deviation **0.05**) and the tighter `APPROX_COUNT_DISTINCT(col, 0.01)` variant on 100K rows.
3. State the accuracy/memory/speed trade-off interviewers expect: error ε needs O(1/ε²) sketch space per group; the default 5% rsd is usually plenty.

Verify: both approximate results are within their rsd budget of the exact count (5,000 distinct users).

In [0]:
# Solution Q2 — exact vs approximate distinct count
import pyspark.sql.functions as F

spark.sql("DROP TABLE IF EXISTS workspace.default.d14_users")
users = (spark.range(0, 100000)
         .withColumn("user_id", (F.rand(seed=42) * 5000).cast("int")))
users.write.mode("overwrite").saveAsTable("workspace.default.d14_users")

q2 = spark.sql("""
SELECT COUNT(DISTINCT user_id)            AS exact_distinct,
       APPROX_COUNT_DISTINCT(user_id)      AS approx_default,   -- rsd = 0.05
       APPROX_COUNT_DISTINCT(user_id, 0.01) AS approx_tight     -- rsd = 0.01
FROM workspace.default.d14_users
""")
q2.show()

r2 = q2.first()
err_default = abs(float(r2.exact_distinct) - float(r2.approx_default)) / float(r2.exact_distinct)
err_tight   = abs(float(r2.exact_distinct) - float(r2.approx_tight)) / float(r2.exact_distinct)
print(f"default rsd=0.05 → observed error {err_default:.5f} (budget 0.10 ≈ 2σ)")
print(f"tight   rsd=0.01 → observed error {err_tight:.5f} (budget 0.05 ≈ 2σ)")
assert err_default < 0.10 and err_tight < 0.05
print("Q2 OK")

# Interview talk-track:
# 1. EXACT COUNT(DISTINCT) — Spark must shuffle every distinct key to an aggregator
#    per group; with billions of keys that is a huge shuffle + memory risk. Sketches
#    avoid it entirely.
# 2. HLL — each worker builds a fixed-size sketch (sparse registers) over its
#    partition; sketches MERGE cheaply (max per register) and estimate cardinality
#    from register statistics. No raw keys ever cross the network.
# 3. Trade-off — accuracy ε costs O(1/ε²) sketch space: 5% error ≈ a few KB per
#    group, 1% error ≈ ~25x that. Default 0.05 is fine for dashboards; use 0.01
#    for billing-adjacent use.
# 4. Databricks also exposes HLL_SKETCH / HLL_SKETCH_ESTIMATE / HLL_UNION for
#    pre-aggregated sketches stored per day — merge daily sketches for a monthly
#    DAU without re-reading raw events.

## Q3 [INTERMEDIATE] Python — Flatten Nested List: Iterator + Generator

**Scenario**: Implement a `NestedIterator` class that lazily flattens an arbitrarily nested list of ints and lists (LeetCode 341 style) — the data engineer's "flatten schema on demand".

**Requirements**:
1. `NestedIterator(nested)` supports `__iter__`/`__next__` and must be **lazy** — do not flatten everything up front (the real list may be huge or partially built).
2. Also provide a `flatten_gen()` generator version using `yield from`.
3. Handle edge cases: empty inner lists `[]`, deep nesting `[1,[2,[3,[4]],5],[],6]`, and starting on a nested element.

Test: both implementations must yield `[1, 2, 3, 4, 5, 6]` for that input, and `next(NestedIterator([[7]])) == 7`.

In [0]:
# Solution Q3 — explicit-stack iterator + recursive generator
class NestedIterator:
    """Flatten an arbitrarily nested list lazily via a stack of active iterators."""
    def __init__(self, nested):
        self._stack = [iter(nested)]          # stack of live iterators
    def __iter__(self):
        return self
    def __next__(self):
        while self._stack:                    # loop until a value surfaces or stack empties
            try:
                item = next(self._stack[-1])
            except StopIteration:
                self._stack.pop()             # exhausted this level → go back up
                continue
            if isinstance(item, list):
                self._stack.append(iter(item))  # descend — LAZY: nothing flattened yet
            else:
                return item
        raise StopIteration

def flatten_gen(nested):
    """Generator variant — same semantics, one line of recursion."""
    for item in nested:
        if isinstance(item, list):
            yield from flatten_gen(item)      # delegate to sub-generator
        else:
            yield item

# ---- Tests ----
data = [1, [2, [3, [4]], 5], [], 6]
assert list(NestedIterator(data)) == [1, 2, 3, 4, 5, 6]
assert list(flatten_gen(data)) == [1, 2, 3, 4, 5, 6]
assert next(NestedIterator([[7]])) == 7
assert list(NestedIterator([])) == []
print("Q3 OK — iterator and generator agree:", list(NestedIterator(data)))

# Interview notes:
# - Laziness: the stack only holds *iterators*, so memory is O(depth), not O(n) —
#   you can flatten an infinitely nested / lazily-grown stream.
# - `yield from` IS the elegant recursive solution; the stack version is the
#   interview-safe one because it shows you understand next()/StopIteration mechanics.
# - Real-world echo: PySpark's explode() does exactly this for nested arrays/structs.

## Q4 [ADVANCED] Python — Dijkstra Shortest Path with a Min-Heap

**Scenario**: Given a weighted directed graph (service mesh / delivery network), find the cheapest path from a start node to every other node.

**Requirements**:
1. Implement **Dijkstra's algorithm** with `heapq` — lazy-deletion pattern: push duplicates, skip stale entries.
2. Return both `distances` and enough back-pointers (`prev`) to reconstruct the full path to any node.
3. Explain complexity: **O((V+E) log V)** with the heap; why negative weights break it (greedy finality) and what to use instead (Bellman-Ford / SPFA).

Test graph (verify): shortest path A→E is `A→C→B→D→E` with cost **14** (not the direct-ish A→B→D→E = 15).

```text
A→B(4), A→C(2), B→D(5), C→B(1), C→D(8), D→E(6)
```

In [0]:
# Solution Q4 — Dijkstra with heapq (lazy deletion)
import heapq

def dijkstra(graph, start):
    """graph: {node: [(neighbor, weight), ...]} → (dist dict, prev dict)."""
    dist = {start: 0}
    prev = {}                 # back-pointers for path reconstruction
    pq = [(0, start)]         # min-heap of (distance-so-far, node)
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist.get(u, float("inf")):   # LAZY DELETION: stale entry, skip
            continue
        for v, w in graph.get(u, []):
            nd = d + w
            if nd < dist.get(v, float("inf")):           # found a better route
                dist[v] = nd
                prev[v] = u
                heapq.heappush(pq, (nd, v))
    return dist, prev

def shortest_path(graph, start, end):
    dist, prev = dijkstra(graph, start)
    if end not in dist:
        return None, float("inf")
    path, node = [end], end
    while node != start:      # walk back-pointers, then reverse
        node = prev[node]
        path.append(node)
    return path[::-1], dist[end]

# ---- Test ----
graph = {
    "A": [("B", 4), ("C", 2)],
    "B": [("D", 5)],
    "C": [("B", 1), ("D", 8)],
    "D": [("E", 6)],
    "E": [],
}
path, cost = shortest_path(graph, "A", "E")
print("path:", path, "cost:", cost)
assert path == ["A", "C", "B", "D", "E"] and cost == 14, (path, cost)
assert shortest_path(graph, "E", "A")[0] is None          # unreachable direction
print("Q4 OK")

# Interview notes:
# - Lazy deletion beats a decrease-key heap implementation in Python (heapq has no
#   decrease-key); duplicates just sit in the heap and get skipped on pop.
# - Complexity O((V+E) log V); each edge relaxation is a heap push (log V).
# - Negative edges break Dijkstra because a popped node's distance is treated FINAL —
#   a later negative edge could improve it. Use Bellman-Ford O(V·E) for that.
# - Data-engineering echo: routing data-lineage impact analysis, task scheduling
#   with costs, network latency budgets between regions.

## Q5 [INTERMEDIATE] PySpark Coding — Array Columns: transform / aggregate / flatten / array_distinct

**Scenario**: An `orders` table stores line items as an **array of strings** `"SKU:price"` — a very common denormalized landing-zone shape. Analysts need order totals and per-customer distinct SKU counts **without exploding to row-per-item**.

**Requirements**:
1. Compute `order_total` per order using higher-order functions — `TRANSFORM` the array to prices, then `AGGREGATE` to a sum (stay array-side, no explode).
2. Per customer, count **distinct SKUs** across all orders: `FLATTEN` + `ARRAY_DISTINCT` + `SIZE` over a `COLLECT_LIST`.
3. Explain when to stay array-side (small arrays, wide dashboards) vs when to `EXPLODE` into a line-items table (large arrays, SQL analysts).

Verify: order totals are 61.0 / 99.99 / 65.0; alice has 3 distinct SKUs (SKU1, SKU2, SKU4) and bob 1.

In [0]:
# Solution Q5 — higher-order functions on arrays
import pyspark.sql.functions as F

spark.sql("DROP TABLE IF EXISTS workspace.default.d14_orders")
orders_data = [
    (1001, "alice", ["SKU1:25.50", "SKU2:10.00", "SKU1:25.50"]),  # duplicate SKU in one order
    (1002, "bob",   ["SKU3:99.99"]),
    (1003, "alice", ["SKU2:10.00", "SKU4:55.00"]),
]
orders = spark.createDataFrame(orders_data, ["order_id", "customer", "items"])
orders.write.mode("overwrite").saveAsTable("workspace.default.d14_orders")

# 1) order totals WITHOUT explode: TRANSFORM parses prices, AGGREGATE sums them
q5 = spark.sql("""
SELECT order_id, customer,
       AGGREGATE(TRANSFORM(items, i -> CAST(SPLIT(i, ':')[1] AS DOUBLE)), 0D,
                 (acc, x) -> acc + x) AS order_total
FROM workspace.default.d14_orders
ORDER BY order_id
""")
q5.show()

# 2) distinct SKUs per customer across ALL their orders
q5b = spark.sql("""
SELECT customer,
       SIZE(ARRAY_DISTINCT(FLATTEN(COLLECT_LIST(items)))) AS distinct_skus,
       SIZE(FLATTEN(COLLECT_LIST(items))) AS total_items
FROM workspace.default.d14_orders
GROUP BY customer
""")
q5b.show()

rows = {r.order_id: r.order_total for r in q5.collect()}
assert abs(rows[1001] - 61.0) < 1e-9 and abs(rows[1002] - 99.99) < 1e-9 and abs(rows[1003] - 65.0) < 1e-9
r5 = {r.customer: r.distinct_skus for r in q5b.collect()}
assert r5["alice"] == 3 and r5["bob"] == 1
print("Q5 OK")

# Interview notes:
# - Higher-order functions (TRANSFORM / AGGREGATE / FILTER) run per-row inside the
#   engine — no explode/reshape shuffle, no UDF serialization cost.
# - PySpark equivalents: F.transform, F.aggregate, F.array_distinct, F.flatten.
# - Explode when arrays get big (100s+ of items) or downstream consumers need
#   line-item grain SQL: LATERAL VIEW EXPLODE / F.explode into a proper fact table.

## Q6 [ADVANCED] PySpark Coding — Schema Evolution: mergeSchema, overwriteSchema, Column Mapping

**Scenario**: A bronze Delta table ingests vendor files. New files keep arriving with an extra column (`city`). Downstream readers must never break, and you must explain the **safe vs dangerous** ways to evolve a Delta table's schema in code.

**Requirements**:
1. Demonstrate additive evolution: append a 3-column DataFrame to a 2-column Delta table using `.option("mergeSchema", "true")` — old rows backfill NULL for `city`.
2. Explain why `.option("overwriteSchema", "true")` is DANGEROUS (replaces schema outright, can drop columns, rewrites history) and when it's legitimate (full-refresh pipelines with complete new data each run).
3. Cover the production guardrails: `delta.columnMapping.mode = name` (allows renames/drops), `delta.autoMerge` table property, and schema-on-write vs schema-on-read (VARIANT at bronze).

Verify: the table ends with columns `id, name, city`; row 1 has NULL city, row 2 'Pune'.

In [0]:
# Solution Q6 — additive schema evolution on Delta
spark.sql("DROP TABLE IF EXISTS workspace.default.d14_products")
df1 = spark.createDataFrame([(1, "widget")], "id INT, name STRING")
df1.write.mode("overwrite").saveAsTable("workspace.default.d14_products")

# new batch arrives with an EXTRA column
df2 = spark.createDataFrame([(2, "gadget", "Pune")], "id INT, name STRING, city STRING")

try:
    df2.write.mode("append").saveAsTable("workspace.default.d14_products")
    print("plain append succeeded (delta.autoMerge may already be on)")
except Exception as e:
    print("plain append rejected as expected:", type(e).__name__)
    df2.write.mode("append").option("mergeSchema", "true").saveAsTable("workspace.default.d14_products")
    print("mergeSchema append succeeded")

res = spark.sql("SELECT * FROM workspace.default.d14_products ORDER BY id")
res.show()
assert set(res.columns) == {"id", "name", "city"} and res.count() == 2
assert res.filter("id = 1").first().city is None      # old row backfilled NULL
print("Q6 OK — schema evolved to:", res.columns)

# Interview talk-track (say this out loud):
# 1. ADDITIVE evolution -> mergeSchema=true: new columns appear, old rows read NULL.
#    Safe for append-only bronze/silver. Table-wide default: ALTER TABLE ... SET TBLPROPERTIES
#    ('delta.autoMerge' = 'true').
# 2. overwriteSchema=true -> rewrites the ENTIRE schema (must re-supply all data).
#    DANGEROUS on shared tables: silently drops columns missing from the DataFrame and
#    invalidates concurrent readers. Legit for full-refresh pipelines with a complete
#    new dataset each run (partition-overwrite staging pattern).
# 3. RENAME / DROP / TYPE changes -> require delta.columnMapping.mode = 'name' on UC tables
#    (Databricks sets it automatically). After enabling it, ALTER TABLE ... RENAME COLUMN
#    works without rewriting files. Trade-off: column mapping BREAKS streaming reads that
#    predate the change and requires Delta reader v2+.
# 4. Schema-on-read alternative: land raw JSON as VARIANT, evolve at query time —
#    you trade strict contracts for flexibility at the bronze layer.

## Q7 [INTERMEDIATE] Spark Optimization — The Small-Files Problem & File Sizing

**Scenario**: A pipeline writes every 5 minutes with `repartition(16)`; queries on the resulting table crawl. `DESCRIBE DETAIL` shows thousands of tiny files.

**Requirements**:
1. Explain why many small files hurt (metadata pressure in the log, open-file overhead per task, scan planning, longer `OPTIMIZE`/`VACUUM` windows) and where small files typically come from (over-partitioning, high-frequency streaming micro-batches).
2. Demonstrate the diagnosis loop: create a 16-small-file table, check `numFiles`/`sizeInBytes` via `DESCRIBE DETAIL`, run `OPTIMIZE`, verify compaction.
3. State the prevention toolkit: `repartition(n)` targeting **128MB-1GB** files, `coalesce` to *reduce* partitions safely (no shuffle), `delta.targetFileSize`, `delta.autoOptimize.optimizeWrite` / optimized writes, and scheduled `OPTIMIZE`.

Verify: 16 files before OPTIMIZE → 1 file after.

In [0]:
# Solution Q7 — diagnose & fix small files (live demo on a tiny table)
import pyspark.sql.functions as F

spark.sql("DROP TABLE IF EXISTS workspace.default.d14_smallfiles")
# simulate a bad writer: 10K rows spread across 16 tiny files
(spark.range(0, 10000)
 .withColumn("bucket", F.pmod(F.col("id"), F.lit(16)))
 .repartition(16)
 .write.mode("overwrite")
 .saveAsTable("workspace.default.d14_smallfiles"))

before = spark.sql("DESCRIBE DETAIL workspace.default.d14_smallfiles").select("numFiles", "sizeInBytes").first()
print(f"BEFORE OPTIMIZE: numFiles={before.numFiles}, sizeInBytes={before.sizeInBytes}")

spark.sql("OPTIMIZE workspace.default.d14_smallfiles")   # bin-packing compaction

after = spark.sql("DESCRIBE DETAIL workspace.default.d14_smallfiles").select("numFiles", "sizeInBytes").first()
print(f"AFTER  OPTIMIZE: numFiles={after.numFiles}, sizeInBytes={after.sizeInBytes}")
assert before.numFiles >= 10 and after.numFiles < before.numFiles
print("Q7 OK — compaction reduced files", before.numFiles, "→", after.numFiles)
spark.sql("DROP TABLE IF EXISTS workspace.default.d14_smallfiles")

# Interview talk-track:
# 1. WHY small files hurt: every Delta file is a log entry + a scan task. 100k tiny files
#    -> slow snapshot parsing, thousands of trivial tasks (open/close overhead dominates
#    actual read), longer vacuum/optimize windows, driver coordination cost.
# 2. WHERE they come from: over-repartitioning (repartition(200) for a 50MB result),
#    micro-batch streaming sinks, per-partition overwrites.
# 3. PREVENTION, in order of preference:
#    - Optimized writes: TBLPROPERTIES ('delta.autoOptimize.optimizeWrite'='true') or
#      spark.databricks.delta.optimizeWrite.enabled=true — the engine sizes files for you.
#    - Pick repartition(n) with n ≈ total_bytes / target_file_size (128MB-1GB);
#      use coalesce(n) to REDUCE partitions without a shuffle (never coalesce to INCREASE).
#    - delta.targetFileSize table property to raise/lower compaction target.
#    - Scheduled OPTIMIZE (e.g. nightly job) + history retention so VACUUM stays cheap.
# 4. On Databricks, prefer liquid clustering over manual partitions — it also lays out
#    files in sensible target-size chunks (see Day 13 for CLUSTER BY).

## Q8 [ADVANCED] Deep Spark — Garbage Collection Tuning & Executor Memory Layout

**Scenario**: A long-running streaming job on classic compute shows 40+ second GC pauses; executors get recycled with `Container killed by YARN for exceeding memory limits`; Spark UI GC time is 30% of task time.

**Requirements** (explain + prescribe):
1. Draw the executor memory map from memory: **heap** (`spark.executor.memory`) split by `spark.memory.fraction` (0.6) into unified execution+storage and user-data regions, vs **off-heap** (`spark.memory.offHeap.enabled/size`) and **overhead** (`spark.executor.memoryOverhead` / `memoryOverheadFactor`, default 0.1-0.15) for JVM netty/direct buffers.
2. Diagnose from symptoms: long Full-GC pauses → heap too small or caching too much; `exceeding memory limits` kills → **overhead** too small (not the heap!).
3. Prescribe fixes: G1GC tuning flags for Spark, right-sizing `memoryOverhead`, unified memory tuning, and when to go off-heap. Note: **serverless compute manages all this for you** — classic clusters only.

In [0]:
# Solution Q8 — GC / memory talk-track with concrete configs
from pyspark import SparkConf

conf = SparkConf()
conf.set("spark.executor.memory", "24g")
conf.set("spark.executor.memoryOverhead", "4g")
conf.set("spark.memory.fraction", "0.6")
conf.set("spark.executor.extraJavaOptions",
         "-XX:+UseG1GC -XX:G1HeapRegionSize=32m "
         "-XX:InitiatingHeapOccupancyPercent=35 -XX:+UnlockDiagnosticVMOptions "
         "-XX:+G1SummarizeConcMark")
print("Reference config (classic clusters):")
for k, v in conf.getAll():
    print(f"  {k} = {v}")
print("Q8 OK — study the talk-track below, then recite it.")

# Interview talk-track — the 5 points that earn the job:
#
# 1. MEMORY MAP (recite): executor total = HEAP (spark.executor.memory) + OVERHEAD
#    (memoryOverhead = heap × 0.1-0.15, covers netty/direct/JVM internals — anything
#    off-heap not via spark.memory.offHeap). Inside the heap, spark.memory.fraction=0.6
#    is split ~50/50 execution/storage (unified memory, borrows both ways);
#    the other 40% is user data (UDF objects, row conversion).
#
# 2. SYMPTOM → DIAGNOSIS:
#    - Frequent Full GC / long pauses → heap undersized for cached+shuffle data,
#      or a memory leak in a UDF. Fix: more heap, LESS caching (drop persist'd DFs),
#      or off-heap for big caches.
#    - 'Container killed ... exceeding memory limits' → the CONTAINER (heap+overhead)
#      blew past its YARN allocation. Classic rookie fix: raise executor.memory — WRONG.
#      Raise memoryOverhead; shuffle/netty buffers live there.
#    - GC time 30% of task time → object churn: avoid collect() of big data, prefer
#      DataFrame/Dataset codegen (Tungsten works on binary, few JVM objects).
#
# 3. G1GC FOR SPARK (default on JDK 17+ clusters): region size ~ 1/2 the average
#    humongous object (32m for big shuffles), InitiatingHeapOccupancyPercent 35 so
#    concurrent marking starts before the heap fills. Parallel GC is fine for short-
#    lived batch jobs; G1 wins for long-running streaming.
#
# 4. OFF-HEAP: spark.memory.offHeap.enabled + size lets Tungsten allocate execution/
#    storage outside the JVM — near-zero GC for cached/buffered data, but serialization
#    cost. Use it for big static caches + heavy shuffles on memory-tight clusters.
#
# 5. SERVERLESS CAVEAT: on Databricks serverless / SQL warehouses, memory & GC are
#    managed by the service — this whole question is a CLASSIC-compute topic. Say that
#    in the interview: it shows you know when the knobs matter and when they don't.

## Q9 [INTERMEDIATE] ETL — Auto Loader: Incremental File Ingestion Done Right

**Scenario**: 2,000 CSV/JSON files land daily in a cloud bucket / UC volume. A naive `spark.read` reprocesses everything each run. You propose **Auto Loader** (`cloudFiles` source).

**Requirements**:
1. Write the Auto Loader read (annotated options): `cloudFiles.format`, `cloudFiles.schemaLocation`, `cloudFiles.inferColumnTypes`, `maxFilesPerTrigger`, bad-record handling, and schema evolution mode.
2. Explain the two **file discovery modes** (directory listing vs `cloudFiles.useNotifications` with a queue) and when each is right.
3. Explain **exactly-once file semantics**: the RocksDB file ledger inside the checkpoint means re-runs never double-ingest; `Trigger.AvailableNow` makes it a batch-friendly incremental job.

This is a design/code question — walk the annotated code below and explain each option out loud.

In [0]:
# Solution Q9 — Auto Loader annotated design (study & recite)
AUTO_LOADER_CODE = '''
from pyspark.sql.functions import col, current_timestamp

raw = (spark.readStream
    .format("cloudFiles")                      # Auto Loader source
    .option("cloudFiles.format", "json")        # csv | json | parquet | ...
    .option("cloudFiles.schemaLocation", CHECKPOINT + "_schema")
       # inferred schema is PERSISTED here — consistent across restarts,
       # and new columns are detected against it (evolution)
    .option("cloudFiles.inferColumnTypes", "true")
       # real types (ints/dates) instead of all-strings
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
       # addNewColumns (safe, additive) | rescue (unexpected fields to _rescued_data)
       # | failOnNewColumns | noEvolution
    .option("cloudFiles.useNotifications", "false")
       # false -> directory listing; true -> event/queue notifications
    .option("maxFilesPerTrigger", 500)
       # backpressure: smooth the burst of 2000-file arrivals
    .option("badRecordsPath", BAD_RECORD_PATH)
       # malformed files/rows quarantined instead of failing the stream
    .load("s3://bucket/raw/events/")            # or a UC volume path
    .withColumn("ingest_ts", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path")))

(raw.writeStream
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)                 # process what's there, then stop — batch-style
    .toTable("bronze.events")
    .awaitTermination())
'''
print(AUTO_LOADER_CODE)
print("Q9 OK — recite the talk-track below.")

# Interview talk-track:
# 1. DISCOVERY MODES — directory listing (default): driver lists the source incrementally,
#    fine up to tens of thousands of files; file notifications: cloud queue (SQS/EventGrid/
#    PubSub) pushes arrival events, scales to millions of files/day. Rule of thumb:
#    > ~50k new files/day or listing latency hurting you -> notifications.
# 2. EXACTLY-ONCE FILES — Auto Loader keeps a RocksDB file ledger inside the checkpoint;
#    each file is processed once even across restarts/replays. Files are tracked by path,
#    so DON'T reuse the same names for different content.
# 3. SCHEMA HANDLING — schemaLocation persists inference; evolution modes trade safety
#    (addNewColumns) for rescue-everything (rescue -> _rescued_data column).
# 4. TRIGGER — AvailableNow = "run like a batch, offset by checkpoint"; ProcessingTime
#    for continuous micro-batches. On serverless, AvailableNow pipelines are cheap and
#    composable with Jobs table-update triggers.
# 5. VS read.files()/COPY INTO: Auto Loader scales better (incremental discovery,
#    notifications, schema evolution); COPY INTO is fine for small (<1k files) simple loads.

## Q10 [INTERMEDIATE] ETL — foreachBatch + Idempotent MERGE Sink (Live Demo)

**Scenario**: A streaming pipeline must write upserts into a Delta **table** with `MERGE` (dedupe on business key). Default sinks can't do MERGE — you need `foreachBatch`.

**Requirements**:
1. Write a `foreachBatch` sink that MERGEs each micro-batch into a target table on `id` (update + insert).
2. Make it **idempotent**: dedupe within the batch (`dropDuplicates` on key), and explain why the checkpoint + MERGE on key makes replays safe even if a batch runs twice (at-least-once → effectively-once).
3. Demonstrate: after processing a source with a correction row (id=1 updated), re-running the stream produces **no duplicates** — still 3 target rows.

Verify: target ends with `{1: alice_v2, 2: bob, 3: carol}`; re-run keeps 3 rows.

In [0]:
# Solution Q10 — live foreachBatch idempotent MERGE demo
import pyspark.sql.functions as F

VOL = "/Volumes/workspace/default/etl_test_volume"

# start clean: a stale checkpoint from a PREVIOUS notebook run points at a different
# physical table (d14_src was dropped/recreated -> new table id) and the stream would
# fail with DIFFERENT_DELTA_TABLE_READ_BY_STREAMING_SOURCE. Real-world lesson:
# NEVER reuse a checkpoint location against a rebuilt table -- that error is Delta
# protecting you from silently reading the wrong source.
try:
    dbutils.fs.rm(f"{VOL}/d14_ckpt_q10", recurse=True)
except Exception:
    pass

spark.sql("DROP TABLE IF EXISTS workspace.default.d14_src")
spark.sql("DROP TABLE IF EXISTS workspace.default.d14_tgt")
spark.sql("CREATE TABLE workspace.default.d14_src (id INT, name STRING, updated_at TIMESTAMP) USING DELTA")

(spark.createDataFrame([(1, "alice", "2026-10-26 09:00:00"), (2, "bob", "2026-10-26 09:00:00")],
      "id INT, name STRING, updated_at STRING")
 .withColumn("updated_at", F.to_timestamp("updated_at"))
 .write.mode("append").saveAsTable("workspace.default.d14_src"))
# a correction arrives later in the log
(spark.createDataFrame([(1, "alice_v2", "2026-10-26 10:00:00"), (3, "carol", "2026-10-26 10:00:00")],
      "id INT, name STRING, updated_at STRING")
 .withColumn("updated_at", F.to_timestamp("updated_at"))
 .write.mode("append").saveAsTable("workspace.default.d14_src"))

# bootstrap empty target, then the idempotent MERGE sink
def merge_batch(df, batch_id):
    df.createOrReplaceTempView("d14_updates")
    spark.sql("""
      MERGE INTO workspace.default.d14_tgt t
      USING d14_updates s ON t.id = s.id
      WHEN MATCHED THEN UPDATE SET *
      WHEN NOT MATCHED THEN INSERT *
    """)

spark.sql("CREATE TABLE workspace.default.d14_tgt AS SELECT * FROM workspace.default.d14_src WHERE 1=0")

for run in (1, 2):   # second run simulates a re-run / replay
    q = (spark.readStream.table("workspace.default.d14_src")
         .dropDuplicates(["id"])                        # dedupe WITHIN a batch
         .writeStream
         .foreachBatch(merge_batch)
         .option("checkpointLocation", f"{VOL}/d14_ckpt_q10")
         .trigger(availableNow=True)                    # process available data, stop
         .start())
    q.awaitTermination()

tgt = spark.table("workspace.default.d14_tgt").orderBy("id")
tgt.show()
rows = {r.id: r.name for r in tgt.collect()}
assert rows == {1: "alice_v2", 2: "bob", 3: "carol"}, rows
assert spark.table("workspace.default.d14_tgt").count() == 3   # replay added nothing
print("Q10 OK — idempotent: replay produced no duplicates")

# Interview notes:
# - Streaming delivers AT-LEAST-ONCE: a failed-then-retried batch can reprocess rows.
#   MERGE on the business key makes re-application a no-op → effectively-once in the sink.
# - dropDuplicates BEFORE the MERGE guards duplicates inside one batch; the checkpoint
#   (offsets, here in a UC volume) guards across runs. Both layers are needed.
# - foreachBatch runs on a static view per micro-batch, so you can use ANY batch API
#   inside it — MERGE, replaceWhere overwrites, multi-table fan-out — at the cost of
#   losing some stream-source optimizations (e.g. exactly-once sink guarantees).

## Q11 [ADVANCED] ETL — Delta Optimistic Concurrency & MERGE Conflict Retry

**Scenario**: Two pipelines upsert the **same Delta table** concurrently (a streaming silver job and an hourly backfill). Some MERGEs fail with `ConcurrentModificationException`. Fix the architecture and the code.

**Requirements**:
1. Explain Delta's **optimistic concurrency control (OCC)**: writers never lock; at commit time each writer checks whether its read snapshot (version) is still the latest — conflicting logical operations (same files rewritten / ambiguous row predicates) get rejected, non-conflicting ones both commit.
2. Classify conflicts: which concurrent writes are **safe** (e.g. append+append, MERGEs with provably disjoint predicates) vs **unsafe** (two MERGEs rewriting the same files).
3. Implement a **retry-with-backoff wrapper** for MERGE that catches `ConcurrentModificationException` and retries with exponential backoff + jitter, and name the architectural fixes that eliminate conflicts in the first place (partition/liquid-cluster the predicate, separate tables + scheduled window, stagger schedules).

In [0]:
# Solution Q11 — OCC explainer + production retry wrapper (runnable, simulated)
import time, random

class ConcurrentModificationException(Exception):
    """Stand-in for io.delta.exceptions.ConcurrentModificationException (illustrative)."""

# ---- the retry wrapper: catch OCC, back off exponentially, recompute the MERGE ----
def merge_with_retry(target, updates_df, on_cols, max_retries=5, base_ms=500):
    """
    MERGE with OCC-safe retry. The TARGET SNAPSHOT is re-read on each attempt —
    Delta MERGE re-resolves the target inside .execute(); the UPDATES side must be
    deterministic (or recomputed) so replay is safe.
    """
    for attempt in range(max_retries + 1):
        try:
            (target.alias("t")
                  .merge(updates_df.alias("s"), " AND ".join(f"t.{c} = s.{c}" for c in on_cols))
                  .whenMatchedUpdateAll()
                  .whenNotMatchedInsertAll()
                  .execute())
            return attempt
        except ConcurrentModificationException:
            if attempt == max_retries:
                raise
            sleep_ms = base_ms * (2 ** attempt) + random.uniform(0, 250)  # exp backoff + jitter
            print(f"  OCC conflict on attempt {attempt}; retrying in {sleep_ms:.0f} ms")
            time.sleep(sleep_ms / 1000)

# ---- simulate a table where the first MERGE attempt loses the OCC race ----
spark.sql("DROP TABLE IF EXISTS workspace.default.d14_occ")
spark.sql("CREATE TABLE workspace.default.d14_occ (id INT, val STRING) USING DELTA")
spark.sql("INSERT INTO workspace.default.d14_occ VALUES (1, 'old')")

real_merge_attempts = {"n": 0}
def flaky_first_attempt(df):
    """Wrap updates so the first execution raises OCC (simulating a racing writer)."""
    real_merge_attempts["n"] += 1
    if real_merge_attempts["n"] == 1:
        raise ConcurrentModificationException("simulated: another writer committed first")
    return df

updates = spark.createDataFrame([(1, "new")], "id INT, val STRING")
class FlakyTable:
    def alias(self, *a):
        class _M:
            def merge(inner_self, df, cond):
                class _Merge:
                    def __init__(s): s._df = df
                    def whenMatchedUpdateAll(s): return s
                    def whenNotMatchedInsertAll(s): return s
                    def execute(s):
                        flaky_first_attempt(s._df)          # may raise OCC on attempt 1
                        spark.sql("MERGE INTO workspace.default.d14_occ t "
                                  "USING d14_occ_upd s ON t.id = s.id "
                                  "WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *")
                return _Merge()
        return _M()

updates.createOrReplaceTempView("d14_occ_upd")
attempts_used = merge_with_retry(FlakyTable(), updates, ["id"], base_ms=10)
final = spark.table("workspace.default.d14_occ").collect()
assert attempts_used == 1 and final == [(1, "new")], (attempts_used, final)
print("Q11 OK — OCC retry wrapper recovered on attempt 1, final value:", final)

# Interview talk-track:
# 1. OCC: Delta has NO locks. Each writer: (a) reads snapshot vN, (b) stages writes,
#    (c) at COMMIT checks the table is still at vN. If not, Delta checks whether the
#    winner's operation CONFLICTS logically — appends never conflict with each other;
#    two MERGEs pass only when they touch DISJOINT files (partition/cluster pruning
#    proves it); anything ambiguous -> ConcurrentModificationException for the loser.
# 2. WHY MERGEs clash: both rewrite the files containing matched rows. Partition the
#    table (or liquid-cluster) on the MERGE key so concurrent writers touch different
#    files -> conflicts drop to near zero.
# 3. RETRY: exponential backoff + jitter (jitter avoids synchronized thundering
#    retries). The MERGE re-resolves the target snapshot each attempt, so data
#    correctness is preserved; only ORDER guarantees are lost (last-writer-wins/row).
# 4. ARCHITECTURE first, retry second: stagger schedules, split hot tables by domain,
#    isolate streaming vs backfill into different phases (Run If dependencies in
#    Jobs), and only then accept occasional retries.

## Q12 [INTERMEDIATE] Data Modeling — Junk Dimensions & Role-Playing Dimensions

**Scenario**: An order fact table has 8 low-cardinality yes/no + status flags (`is_gift`, `is_mobile`, `payment_status`, `fraud_flag`, ...). Adding each as a column works but creates a wide, sparse fact. Also, the fact needs **order_date**, **ship_date**, and **return_date** — but there is only one `dim_date`.

**Requirements**:
1. Explain **junk dimensions**: collapse many low-cardinality flags/statuses into ONE dimension whose rows are the observed combinations of values; fact stores a single surrogate key. Explain the gains (narrower fact, fewer joins) and the caveats (cartesian explosion if unmanaged combos, keep it under ~1M rows).
2. Explain **role-playing dimensions**: one physical `dim_date`, referenced multiple times in the fact via views/aliases (`order_date_key`, `ship_date_key`, `return_date_key`); why you must NOT copy the dimension three times.
3. Write the DDL for a junk dimension builder (generate distinct flag combos + surrogate keys) and the role-playing date references.

Design question — walk the DDL and reasoning in the solution cell.

In [0]:
# Solution Q12 — junk + role-playing dimension design
JUNK_DDL = """
-- JUNK DIMENSION: one row per OBSERVED flag combination, surrogate-keyed.
-- Built incrementally: new combos in staging get new keys via MERGE (SCD-1).
CREATE TABLE silver.dim_order_flags (
  order_flags_key BIGINT,       -- surrogate (hash or sequence)
  is_gift        BOOLEAN,
  is_mobile      BOOLEAN,
  gift_wrap_type STRING,         -- tiny domain: none|standard|deluxe
  payment_status STRING,         -- authorized|captured|refunded
  fraud_flag     BOOLEAN
) USING DELTA;

-- Builder: distinct combos that don't exist yet -> insert with new keys
MERGE INTO silver.dim_order_flags d
USING (SELECT DISTINCT is_gift, is_mobile, gift_wrap_type, payment_status, fraud_flag
       FROM bronze.order_flags_staging) s
ON  d.is_gift = s.is_gift AND d.is_mobile = s.is_mobile
AND d.gift_wrap_type <=> s.gift_wrap_type AND d.payment_status <=> s.payment_status
AND d.fraud_flag = s.fraud_flag
WHEN NOT MATCHED THEN INSERT
  (order_flags_key, is_gift, is_mobile, gift_wrap_type, payment_status, fraud_flag)
  VALUES (xxhash64(concat_ws('|', s.is_gift, s.is_mobile,
           coalesce(s.gift_wrap_type,'~'), coalesce(s.payment_status,'~'), s.fraud_flag)),
          s.is_gift, s.is_mobile, s.gift_wrap_type, s.payment_status, s.fraud_flag);

-- FACT: ONE junk key replaces 5-8 flag columns (narrower rows, one join to get all flags)
CREATE TABLE silver.fact_orders (
  order_id        BIGINT,
  order_flags_key BIGINT,        -- junk dimension reference
  order_date_key  INT,           -- role-playing date keys: same physical dim_date
  ship_date_key   INT,
  return_date_key INT,
  customer_key    BIGINT,
  amount          DECIMAL(12,2)
) USING DELTA;

-- ROLE-PLAYING: three logical views over ONE dim_date — no copies, no drift
CREATE OR REPLACE VIEW silver.v_fact_orders_ordered AS
  SELECT f.*, d.date_actual AS order_date FROM silver.fact_orders f JOIN silver.dim_date d ON f.order_date_key = d.date_key;
CREATE OR REPLACE VIEW silver.v_fact_orders_shipped AS
  SELECT f.*, d.date_actual AS ship_date  FROM silver.fact_orders f JOIN silver.dim_date d ON f.ship_date_key  = d.date_key;
CREATE OR REPLACE VIEW silver.v_fact_orders_returned AS
  SELECT f.*, d.date_actual AS return_date FROM silver.fact_orders f JOIN silver.dim_date d ON f.return_date_key = d.date_key;
"""
print(JUNK_DDL)
print("Q12 OK — recite the design reasoning below.")

# Interview talk-track:
# 1. JUNK DIMENSION — why: 8 boolean/status columns in the fact = 8 narrow-but-slow
#    columns that analysts never filter individually; as a dimension, the fact carries
#    ONE bigint key and flags are looked up in a single small join. Kimball's rule:
#    low-cardinality, mutually-independent-ish flags -> junk them.
#    HOW: distinct-combo rows, deterministic surrogate (xxhash64 of combo -> idempotent
#    builds across reruns), MERGE-only inserts. CAVEAT: the theoretical cross-product of
#    the full domains can explode (2^8+ combos x statuses); you only store OBSERVED combos,
#    so cap the dimension and monitor growth — if it exceeds ~1M rows or a flag domain
#    grows, promote that flag to its own dimension.
# 2. ROLE-PLAYING DIMENSION — one physical dim_date, multiple FK columns in the fact,
#    one view per role. Never duplicate the dimension (maintenance drift, 3x storage,
#    inconsistent fiscal attributes). Snowflake/BI tools handle this with aliases too.
# 3. EXTRAS interviewers love: degenerate dims (order_number stored in fact, no dim —
#    Day 12), and mini-dimensions for rapidly-changing attributes (Q13 today).

## Q13 [ADVANCED] Data Modeling — SCD Type 3, Type 4, and Type 6 (Live Type-3 MERGE)

**Scenario**: You already know SCD Type 1 (overwrite) and Type 2 (full history rows). The interviewer pushes further: what about Type 3, 4, and 6?

**Requirements**:
1. **Type 3 (previous-value)**: store `current_price` + `previous_price` (+ effective dates) in the SAME row. Implement the MERGE: on a price change, old current becomes previous, new arrives. When is Type 3 the right choice (only one level of history needed, e.g. price-trend or last-known-address analyses)?
2. **Type 4 (mini-dimension)**: for RAPIDLY changing attributes (e.g. customer credit score changing daily), split the fast attribute into a separate mini-dim with its own surrogate keys; the fact references the mini-dim key at event time, not the customer. Explain why SCD-2 would explode the customer dim.
3. **Type 6 (hybrid)**: SCD2 + SCD1 + SCD3 combined — history rows (Type 2), an `is_current` flag (Type 1-style fast lookup), and previous-value columns (Type 3) so the current row alone can answer both "now" and "what changed last".

Verify (live): product 1's price change 25→30 moves 25 into `previous_price`; unchanged product 2 stays 99/95; new product 3 inserts with NULL previous.

In [0]:
# Solution Q13 — live Type-3 MERGE + Type-4/Type-6 design
spark.sql("DROP TABLE IF EXISTS workspace.default.d14_products_scd")
spark.sql("""
CREATE TABLE workspace.default.d14_products_scd (
  product_id INT, name STRING, current_price DOUBLE, previous_price DOUBLE,
  price_effective_from DATE, price_effective_to DATE
) USING DELTA
""")
spark.sql("""
INSERT INTO workspace.default.d14_products_scd VALUES
 (1, 'widget', 25.0, 20.0, DATE'2026-01-01', DATE'2026-06-30'),
 (2, 'gadget', 99.0, 95.0, DATE'2026-02-01', DATE'2026-06-30')
""")

new_prices = spark.createDataFrame([(1, 30.0), (3, 50.0)], "product_id INT, new_price DOUBLE")
new_prices.createOrReplaceTempView("d14_new_prices")

# TYPE 3: single row per product — old current demoted to previous, new price applied
spark.sql("""
MERGE INTO workspace.default.d14_products_scd t
USING d14_new_prices s
ON t.product_id = s.product_id
WHEN MATCHED AND t.current_price <> s.new_price THEN UPDATE SET
  previous_price = t.current_price,     -- demote
  current_price  = s.new_price,         -- promote
  price_effective_to = current_date()    -- close the old window
WHEN NOT MATCHED THEN INSERT
  (product_id, name, current_price, previous_price, price_effective_from, price_effective_to)
  VALUES (s.product_id, 'unknown', s.new_price, NULL, current_date(), NULL)
""")
res = spark.table("workspace.default.d14_products_scd").orderBy("product_id")
res.show()
r13 = {r.product_id: r for r in res.collect()}
assert r13[1].current_price == 30.0 and r13[1].previous_price == 25.0
assert r13[2].current_price == 99.0 and r13[2].previous_price == 95.0   # untouched
assert r13[3].current_price == 50.0 and r13[3].previous_price is None
print("Q13 OK — Type-3 MERGE verified")

# TYPE 4 (mini-dimension) design — say this out loud:
# Customer credit_score changes DAILY. SCD-2 would mint a new customer row per change:
# 10M customers x 365 changes/yr = 3.65B dim rows/year. Absurd.
# Instead: dim_customer (stable attributes, SCD-1/2) + dim_customer_credit_mini
# (banded values: score 700-750, 750-800, ...) with its own surrogate keys.
# The FACT references (customer_key, credit_mini_key) AT EVENT TIME — analysis joins
# fact -> mini-dim directly, giving 'score at the moment of the transaction' for free.

# TYPE 6 (hybrid) — the one-row-says-everything pattern:
# Columns: [surrogate, natural_key, attr..., valid_from, valid_to, is_current,
#           previous_<attr>]. History = Type 2 rows; fast current lookup = WHERE
# is_current; last change = previous_* columns (Type 3).
# "6" = 2+1+3. Standard in finance dims (customer address + credit banding).

spark.sql("DROP TABLE IF EXISTS workspace.default.d14_products_scd")
print("cleanup done")

## Q14 [INTERMEDIATE] Snowflake — Micro-Partitions & Partition Pruning

**Scenario**: A 5TB Snowflake query scans 100% of a table even with a `WHERE event_date BETWEEN` filter. The interviewer asks how Snowflake stores data and how pruning works.

**Requirements**:
1. Explain **micro-partitions**: Snowflake's immutable ~16MB compressed (50-250MB uncompressed) units, created automatically on load, each with per-column **metadata stored in the cloud services layer** (min/max values, distinct counts, NULL counts) — this metadata is what powers pruning, cloning, and Time Travel.
2. Explain **pruning**: filters are evaluated against micro-partition metadata to skip partitions at planning time; show the diagnostic loop — `EXPLAIN` reading `Partitions Assigned/Total`, `SYSTEM$PARTITION_STATS`, latency before/after load-order fixes (natural vs. clustered).
3. Contrast with Hive-style partitioning: why explicit partition keys are unnecessary in Snowflake in most cases, and when **clustering keys** take over (Day 13's topic — mention, don't repeat).

Design/explain question — walk the SQL and reasoning in the solution cell.

In [0]:
# Solution Q14 — micro-partition & pruning talk-track with diagnostics
PRUNING_SQL = """
-- 1. See the plan: which partitions get assigned vs total
EXPLAIN SELECT * FROM events
WHERE event_date BETWEEN '2026-10-01' AND '2026-10-07' AND country = 'IN';
-- Read the output: "Partitions assigned: 42, Partitions total: 12000" → 0.35% scanned.
-- If assigned ≈ total → NO pruning → your filter doesn't match the physical layout.

-- 2. Inspect per-partition statistics for the filter column
SELECT * FROM TABLE(SYSTEM$PARTITION_STATS('EVENTS', 'EVENT_DATE'));
-- shows min/max per micro-partition — you can literally see the sort ranges.

-- 3. (If table is large + naturally-unordered) add a clustering key
ALTER TABLE events CLUSTER BY (event_date, country);
-- then re-check EXPLAIN and SYSTEM$CLUSTERING_INFORMATION('EVENTS') (Day 13).
"""
print(PRUNING_SQL)
print("Q14 OK — recite the talk-track below.")

# Interview talk-track:
# 1. STORAGE UNIT — every load writes immutable micro-partitions: ~16MB compressed
#    (~50-250MB raw), columnar-compressed, snowflake-managed. You do NOT choose
#    partitions (no PARTITION BY DDL like Hive). Each micro-partition records column
#    metadata — min/max, distinct-count, NULL-count — in the TABLE METADATA kept by the
#    cloud-services layer.
# 2. PRUNING — at compile time the optimizer compares your WHERE predicates to min/max
#    ranges and drops whole micro-partitions without touching storage. That's why load
#    ORDER matters: if data arrives roughly date-sorted, each micro-partition spans a
#    narrow date range and date filters prune beautifully. Random arrival order → each
#    micro-partition spans the whole range → nothing prunes.
# 3. DIAGNOSTIC LOOP — EXPLAIN (Partitions assigned vs total) → SYSTEM$PARTITION_STATS
#    → fix layout (cluster key, or reload sorted via COPY + ORDER BY) → re-check.
# 4. WHY MICRO (vs Hive big partitions): small units make pruning GRANULAR, make
#    zero-copy cloning cheap (metadata-only snapshot of tiny units), and enable
#    Time Travel by keeping old micro-partition files + metadata for retention days.
# 5. When clustering keys? Very large tables whose natural load order doesn't match
#    common filters. Otherwise let Snowflake's automatic micro-partitioning do its job.

## Q15 [ADVANCED] Snowflake — Streams + Tasks: Native CDC Orchestration

**Scenario**: A Snowflake table `raw_orders` gets batch-loaded several times a day. Downstream `dim_orders` must be refreshed incrementally within minutes of each load — no Kafka, no external scheduler allowed for the data layer.

**Requirements**:
1. Design a **Stream** on `raw_orders` — explain what a stream is (a CDC offset over the table's change metadata: appended / inserted / updated / deleted rows + the change flags `METADATA$ACTION` / `METADATA$ISUPDATE`), and why it consumes **no storage**.
2. Design a **Task** to process the stream: `CREATE TASK ... WHEN SYSTEM$STREAM_HAS_DATA('...')` so it skips idle runs (no warehouse burn), `SCHEDULE`, `USER_TASK_MANAGED_INITIAL_WAREHOUSE_SIZE`, error handling (`ERROR_ON_NONDETERMINISTIC_MERGE`, task suspension after repeated failures, notification integrations), and the task-DAG pattern (`AFTER` other tasks).
3. Write the MERGE-from-stream pattern (dedupe with `ROW_NUMBER` over `METADATA$ROW_ID`), and state the gotcha: the stream offset advances only when the consuming transaction commits; a crashed run leaves the stream ready for retry (at-least-once → MERGE makes it idempotent).

Design question — walk the SQL in the solution cell.

In [0]:
# Solution Q15 — Stream + Task native CDC pipeline
STREAM_TASK_SQL = """
-- 1. The CDC offset: a stream is just a pointer into the table's change metadata.
--    No storage, no cost until queried. It exposes changed rows + change flags.
CREATE OR REPLACE STREAM raw_orders_stream ON TABLE raw_orders;

-- 2. The processor task: runs every 5 minutes BUT only consumes a warehouse when
--    the stream actually has rows (SYSTEM$STREAM_HAS_DATA = skip idle runs).
CREATE OR REPLACE TASK t_dim_orders_refresh
  WAREHOUSE = ETL_XS
  SCHEDULE = '5 MINUTE'
  WHEN SYSTEM$STREAM_HAS_DATA('raw_orders_stream')
  ERROR_ON_NONDETERMINISTIC_MERGE = FALSE   -- tolerate re-processing overlap
AS
MERGE INTO dim_orders d
USING (
  SELECT * FROM (
    SELECT o.*, METADATA$ACTION AS action, METADATA$ISUPDATE AS is_update,
           ROW_NUMBER() OVER (PARTITION BY METADATA$ROW_ID
                              ORDER BY METADATA$UPDATE_SEQUENCE DESC) AS rn
    FROM raw_orders_stream o
  ) WHERE rn = 1            -- last change per row wins
) s
ON d.order_id = s.order_id
WHEN MATCHED AND s.action = 'DELETE' THEN DELETE
WHEN MATCHED THEN UPDATE SET d.status = s.status, d.updated_at = s.updated_at
WHEN NOT MATCHED AND s.action = 'INSERT' THEN
  INSERT (order_id, status, updated_at) VALUES (s.order_id, s.status, s.updated_at);

ALTER TASK t_dim_orders_refresh RESUME;   -- tasks start SUSPENDED by default!

-- 3. Task DAG: chain steps (child runs AFTER parent, shares the schedule)
CREATE TASK t_publish_dashboard AFTER t_dim_orders_refresh AS CALL refresh_dashboard_views();
"""
print(STREAM_TASK_SQL)
print("Q15 OK — recite the talk-track below.")

# Interview talk-track:
# 1. STREAM = offset pointer over micro-partition change metadata. Reading the stream
#    shows rows changed since the stream's last consumed offset; the offset ADVANCES
#    ONLY when the transaction that read it COMMITS. Crash mid-run → offset unmoved →
#    next run re-reads the same changes (at-least-once) → MERGE on order_id absorbs
#    duplicates (idempotent). That's exactly the same exactly-once reasoning as
#    Delta streaming checkpoints (Day 11) and foreachBatch MERGE (Q10 today).
# 2. TASK = scheduler INSIDE Snowflake. Key options to recite:
#    - WHEN SYSTEM$STREAM_HAS_DATA → skip empty runs, save credits.
#    - Warehouse sizing (XS usually enough for stream MERGEs) or serverless tasks with
#      USER_TASK_MANAGED_INITIAL_WAREHOUSE_SIZE.
#    - Tasks auto-SUSPEND after repeated failures (default ~10 consecutive) → wire
#      notification integrations (email/Slack via NOTIFY integrations) + a re-arm runbook.
#    - DAG via AFTER ... — a child task inherits the parent's completion trigger.
# 3. GOTCHAS: one stream = one consumer (one offset); create one stream per consumer.
#    Tasks start suspended. Monitor stream lag via INFORMATION_SCHEMA.STREAMS.
# 4. VS DATABRICKS: this is Snowflake's answer to Delta + Structured Streaming —
#    say the analogy out loud: stream ≈ readStream checkpoint offset, task ≈ mini Job.

## Q16 [INTERMEDIATE] Monitoring & Orchestration — Airflow Scheduling Deep Cuts: catchup, backfill, data intervals

**Scenario**: Your Airflow DAG ran fine for a week, then someone noticed **10 extra runs** appeared overnight, and a `catchup=True` DAG re-ran the whole history into a table with duplicated data. Explain the mechanics.

**Requirements**:
1. Explain the **logical date / data interval** model: a run for 2026-10-25 09:00 UTC actually processes the interval `[09:00, 10:00)` and typically starts ~10:00 — `logical_date` (execution_date) marks the START of the interval, not the run moment.
2. Explain **catchup** (`catchup=True` → scheduler fills every missed interval since the DAG start; `catchup=False` → only latest), and the `catchup` + `max_active_runs` interaction for long queues.
3. Explain **backfill** (`airflow dags backfill -s ... -e ...`) vs re-running single intervals; and why every task must be **idempotent** so catchup/backfill/replay is safe (same input interval → same output, e.g. `replaceWhere` on the interval's partition).
4. Bonus: `depends_on_past`, `wait_for_downstream`, and pause-vs-clear semantics during incident re-runs.

In [0]:
# Solution Q16 — Airflow scheduling model, annotated
AIRFLOW_CODE = '''
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta

with DAG(
    dag_id="hourly_orders",
    schedule="0 * * * *",           # cron — hourly on the hour
    start_date=datetime(2026, 10, 1),
    catchup=False,                 # THE hero of this question:
                                  #   True -> scheduler creates a run for EVERY missed
                                  #   interval since start_date (pause + restart gap,
                                  #   schedule change = surprise runs)
    max_active_runs=2,            # cap concurrent runs when a queue builds up
    default_args={
        "retries": 2,
        "retry_delay": timedelta(minutes=5),
        "depends_on_past": False,  # True -> each run needs the PREVIOUS interval to
                                  # succeed (chain-stop semantics; breaks backfills)
    },
) as dag:
    PythonOperator(
        task_id="load_hour",
        python_callable=load_hour,   # MUST be idempotent on the interval
    )

def load_hour(**context):
    iv = context["data_interval"]    # the interval [start, end) this run owns
    sql = ("COPY INTO target FROM source "
           f"WHERE ts >= '{iv.start}' AND ts < '{iv.end}'")
    # better with Delta: MERGE on business key, or overwrite with
    # .option("replaceWhere", "ts >= ... AND ts < ...") -> re-running is SAFE.
'''
print(AIRFLOW_CODE)
print("Q16 OK — recite the model below.")

# Interview talk-track:
# 1. LOGICAL DATE — a scheduled run is BOUND to a data interval, not a wall-clock
#    moment. The 09:00 run covers [09:00, 10:00) and fires ~10:00. People get burned
#    filtering "yesterday" off logical_date when they wanted the interval END.
#    (Airflow 2.2+ also exposes explicit data_interval_start/end — use those.)
# 2. CATCHUP — scheduler behavior after gaps (paused DAG, changed schedule, late
#    deploy): True -> one run per missed interval, in order, respecting max_active_runs;
#    False -> run only the latest interval. DAILY ETL with idempotent tasks -> catchup
#    True is actually GOOD (auto backfill). Real-time-ish DAGs -> False.
# 3. BACKFILL — CLI/manual: backfill -s START -e END replays historical intervals
#    (use catchup=False during backfills to avoid double-filling). Clearing a task
#    instance re-runs THAT interval only. Both are safe ONLY IF tasks are idempotent:
#    same interval -> same result (MERGE/replaceWhere/DELETE+INSERT window), never
#    blind INSERT. That's why the duplicate-data incident happened: append-only task
#    + surprise catchup = 10 duplicate loads.
# 4. DEPENDS_ON_PAST — sequence guard: run N needs run N-1 success. Sounds safe, but
#    it kills parallel backfills and stalls on any historical failure — prefer
#    idempotency + data checks over chained runs.
# 5. DATABRICKS ANALOGY — Jobs cron triggers + Repair Runs are the equivalent surface
#    (Day 11); say the mapping out loud — logical date ≈ job run's parameterized window.

## Q17 [ADVANCED] Monitoring — Lakehouse Monitoring: Automated Drift & Quality Metrics

**Scenario**: Your team hand-rolls freshness/volume checks into a metrics table (Day 4-5 pattern — fine), but leadership asks for **automated statistical monitoring**: data drift, null-rate spikes, distribution shifts — on 50 silver tables. You propose **Lakehouse Monitoring**.

**Requirements**:
1. Explain the two monitor types: **Snapshot** (profile a table's current state on a schedule; great for batch/silver and dims) vs **TimeSeries** (bucket metrics by a time column + slicing keys; great for streaming/lakehouse facts).
2. What gets produced: a **metrics table family** — slice-level metrics (row counts, null rates, approximate distincts/histograms, min/max), **drift metrics** vs a baseline, and a monitor status/dashboard auto-created per monitor.
3. Write the setup for both types (SQL or the Python API) and state the baseline/drift-window concept and what to alert on.

Design question — walk the code in the solution cell.

In [0]:
# Solution Q17 — Lakehouse Monitoring setup (both types)
MONITOR_SQL = """
-- TimeSeries monitor on a streaming fact: metrics bucketed per event-time slice
CREATE MONITOR silver_orders_ts_monitor
  ON silver.fact_orders
  AS (TIMESERIES timestamp_col = 'order_ts',
      slice_by = ['channel', 'country'],
      schedule = 'CRON 0 0 8 * * ?');

-- Snapshot monitor on a dim: full profile on a cadence, no time dimension
CREATE MONITOR dim_customers_monitor ON silver.dim_customers
  AS (SNAPSHOT, schedule = 'CRON 0 0 6 * * ?');
"""
print(MONITOR_SQL)

PYTHON_API = '''
from lakehouse_monitoring import create_monitor, MonitorType

create_monitor(
    table_name="silver.fact_orders",
    output_schema_name="silver",           # metrics tables land here
    monitoring_type=MonitorType.TIME_SERIES,
    time_column="order_ts",
    slice_columns=["channel", "country"],
    schedule=None,                          # None -> refresh on demand / by job
    baseline_table_name="silver.fact_orders_baseline",  # fixed reference for drift
)
'''
print(PYTHON_API)
print("Q17 OK — recite the talk-track below.")

# Interview talk-track:
# 1. WHAT IT IS — UC-native automated monitoring: scheduled profiling runs write
#    results to a metrics table family: {table}_monitor_metrics, {table}_monitor_drift_metrics,
#    {table}_monitor_profile (slices + status), plus an auto-generated dashboard.
# 2. SNAPSHOT vs TIMESERIES — Snapshot: each run profiles the whole current table
#    (row counts, null rates, approximate histograms per column) — best for dims and
#    slowly-refreshing silvers. TimeSeries: buckets by a time column and slice keys
#    (channel/country) — best for facts; shows per-bucket volume/freshness and how
#    column distributions move over time.
# 3. DRIFT — each slice's distribution is compared to a BASELINE (an earlier slice or
#    an explicit baseline_table) using JS (Jensen-Shannon) distance; drift metrics flag
#    behavioral changes (new category explosion, null-rate spike, numeric range shift)
#    BEFORE they become BI incidents.
# 4. WHAT TO ALERT ON — pair the metrics tables with SQL alerts: slice_status != OK,
#    null_rate > threshold on critical columns, volume vs 7-day average (z-score),
#    drift JS-distance above threshold. Composes with the Day 4/5 metrics-table
#    alerting pattern — same surface, richer signals.
# 5. COST note — snapshot monitors on very large tables re-profile everything each
#    run; prefer TimeSeries with incremental slices, or monitor aggregated silvers.

## Q18 [ADVANCED] Monitoring — SLA/SLO & Burn-Rate Alerting for Data Pipelines

**Scenario**: Leadership asks: "Is the orders pipeline reliable?" You must answer with numbers, not vibes — SRE practice applied to data.

**Requirements**:
1. Define the data-pipeline SLI/SLO: SLI = freshness lag per run (minutes from source commit to silver publish); SLO = **99% of runs meet the 60-minute SLA over 30 days**. State the **error budget**: 1% of runs (≈ 7 misses/month at hourly cadence).
2. Define **burn rate** = actual miss rate ÷ allowed miss rate, and the multi-window multi-burn-rate alerting standard (Google SRE): **14.4×** over 1h → PAGE (2% of monthly budget burned in an hour), **6×** over 6h → ticket, **1×** over 3d → trend watch.
3. Live demo: from a 48-run hourly `run_metrics` history (3 SLA misses), compute achieved availability, the overall burn multiple, the recent (last-6-runs) burn multiple, and the alert action. Verify: 93.75% availability, overall burn 6.25×, recent burn 16.67× → **PAGE**.

**Follow-up**: why is a monthly availability average alone a dangerous alerting signal?

In [0]:
# Solution Q18 — SLA/SLO + burn-rate alerting (live demo)
import datetime as dt

# 48 hourly runs; 3 missed the 60-min freshness SLA (hours 10, 25, 45)
base = dt.datetime(2026, 10, 25, 0, 0)
miss_hours = {10, 25, 45}
rows = []
for h in range(48):
    lag = 95 if h in miss_hours else 18 + (h % 30)
    rows.append(((base + dt.timedelta(hours=h)).strftime("%Y-%m-%d %H:%M:%S"),
                 "orders_silver", int(lag), 60))
(spark.createDataFrame(rows, "run_ts STRING, pipeline STRING, freshness_lag_min INT, sla_min INT")
     .createOrReplaceTempView("run_metrics"))

q18 = spark.sql("""
WITH overall AS (
  SELECT pipeline,
         COUNT(*) AS total_runs,
         SUM(CASE WHEN freshness_lag_min > sla_min THEN 1 ELSE 0 END) AS missed_runs,
         ROUND(100.0 * SUM(CASE WHEN freshness_lag_min > sla_min THEN 1 ELSE 0 END) / COUNT(*), 2) AS miss_pct
  FROM run_metrics
  GROUP BY pipeline
),
recent AS (
  SELECT pipeline, freshness_lag_min, sla_min,
         ROW_NUMBER() OVER (PARTITION BY pipeline ORDER BY run_ts DESC) AS rn
  FROM run_metrics
),
recent_stats AS (
  SELECT pipeline,
         ROUND(100.0 * SUM(CASE WHEN freshness_lag_min > sla_min THEN 1 ELSE 0 END) / COUNT(*), 2) AS recent_miss_pct
  FROM recent
  WHERE rn <= 6
  GROUP BY pipeline
)
SELECT o.pipeline, o.total_runs, o.missed_runs, o.miss_pct,
       ROUND(100 - o.miss_pct, 2)                  AS achieved_availability_pct,
       ROUND(o.miss_pct / (100.0 - 99.0), 2)        AS overall_burn_multiple,   -- SLO 99% -> 1% budget
       r.recent_miss_pct,
       ROUND(r.recent_miss_pct / (100.0 - 99.0), 2) AS recent_burn_multiple,
       CASE WHEN r.recent_miss_pct > 14.4 THEN 'PAGE (fast burn)'
            WHEN r.recent_miss_pct > 6    THEN 'TICKET (slow burn)'
            ELSE 'healthy' END              AS alert_action
FROM overall o
JOIN recent_stats r USING (pipeline)
""")
q18.show(5, False)

row = q18.first()
print(f"achieved availability {row.achieved_availability_pct}% vs SLO 99% -> overall burn {row.overall_burn_multiple}x of monthly budget")
print(f"recent window burn {row.recent_burn_multiple}x -> alert: {row.alert_action}")
assert float(row.miss_pct) == 6.25 and float(row.overall_burn_multiple) == 6.25
assert float(row.recent_miss_pct) == 16.67 and row.alert_action == "PAGE (fast burn)"
print("Q18 OK — burn-rate alerting computed")

# Interview talk-track:
# 1. SLI/SLO TRANSLATED TO DATA — SLI = per-run freshness lag minutes; SLO = 99% of
#    hourly runs under 60 min over 30 days. Error budget = 1% of runs (~7 misses/month).
#    The budget is a license to fail a LITTLE — prioritize reliability work by budget left.
# 2. BURN RATE — actual miss rate / allowed miss rate. At 6.25x, the 30-day budget is
#    spent in ~5 days -> freeze feature work on the pipeline and fix reliability first.
# 3. MULTI-WINDOW MULTI-BURN-RATE (Google SRE standard) —
#      14.4x over 1h AND 14.4x over 5m -> PAGE  (2% of monthly budget in one hour)
#      6x    over 6h AND 6x  over 30m  -> TICKET
#      1x    over 3d                  -> trend watch
#    The two-window AND prevents flapping: the fast window must agree with the slow one.
# 4. WHY NOT MONTHLY AVERAGES — 93.75% monthly availability can hide a cluster of
#    failures happening RIGHT NOW; the recent window is what pages.
# 5. WIRING — a metrics table (one row per run: pipeline, run_ts, freshness_lag_min)
#    + this query as a scheduled SQL alert -> on-call rota. Same pattern composes with
#    the Day 4/5 freshness/volume alerts and Lakehouse Monitoring (Q17).

# 🏁 Day 14 — Score Card

Week 2 complete! Rate yourself per question (1 = struggled, 2 = needed hints, 3 = confident), then review weak spots before Day 15 (Week 3: ADVANCED / HARD product-company level).

| # | Topic | Level | Self (1-3) | Notes |
| --- | --- | --- | --- | --- |
| Q1 | SQL — strict funnel drop-off analysis | INTERMEDIATE |  | |
| Q2 | SQL — approximate distinct count (HLL) | ADVANCED |  | |
| Q3 | Python — nested list iterator + generators | INTERMEDIATE |  | |
| Q4 | Python — Dijkstra shortest path (heapq) | ADVANCED |  | |
| Q5 | PySpark — array higher-order functions | INTERMEDIATE |  | |
| Q6 | PySpark — Delta schema evolution | ADVANCED |  | |
| Q7 | Spark Opt — small files & compaction | INTERMEDIATE |  | |
| Q8 | Spark Opt — GC tuning & memory layout | ADVANCED |  | |
| Q9 | ETL — Auto Loader design | INTERMEDIATE |  | |
| Q10 | ETL — foreachBatch idempotent MERGE | INTERMEDIATE |  | |
| Q11 | ETL — Delta OCC + conflict retry | ADVANCED |  | |
| Q12 | DM — junk & role-playing dimensions | INTERMEDIATE |  | |
| Q13 | DM — SCD Type 3/4/6 | ADVANCED |  | |
| Q14 | Snowflake — micro-partitions & pruning | INTERMEDIATE |  | |
| Q15 | Snowflake — Streams + Tasks CDC | ADVANCED |  | |
| Q16 | Monitoring — Airflow catchup/backfill | INTERMEDIATE |  | |
| Q17 | Monitoring — Lakehouse Monitoring | ADVANCED |  | |
| Q18 | Monitoring — SLA/SLO & burn-rate alerting | ADVANCED |  | |

**Total: ___ / 54** · 18 questions · 8 categories · Week 2, Day 7

**Tomorrow (Day 15)**: Week 3 begins — ADVANCED → [HARD] product-company scenarios, complex system design, and deeper trade-off drills.